## Task 1: Data Understanding
* Load the dataset using Pandas
* Display: First 5 rows, Last 5 rows, Dataset shape, Column names
* Identify: Quantitative data (Discrete & Continuous), Qualitative data (Nominal & Ordinal)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Loading Flights Dataset
df = pd.read_csv('flights.csv')

print("First 5 rows:")
display(df.head())
print("\nLast 5 rows:")
display(df.tail())
print(f"\nDataset shape: {df.shape}")
print(f"Column names: {df.columns.tolist()}")

print("\n--- Data Type Classification ---")
print("Quantitative (Continuous): dep_delay, arr_delay, air_time, distance")
print("Quantitative (Discrete): year, month, day, hour, minute, flight")
print("Qualitative (Nominal): carrier, origin, dest, tailnum, name")
print("Qualitative (Ordinal): None in this dataset")

## Task 2: Exploratory Data Analysis (EDA)
### Univariate Analysis
* Analyze: Departure Delay, Arrival Delay, Distance
* Use: Histogram, Boxplot

In [ ]:
cols = ['dep_delay', 'arr_delay', 'distance']
plt.figure(figsize=(15, 10))
for i, col in enumerate(cols, 1):
    plt.subplot(2, 3, i)
    sns.histplot(df[col].dropna(), kde=True, color='blue')
    plt.title(f'Histogram of {col}')
    
    plt.subplot(2, 3, i+3)
    sns.boxplot(y=df[col].dropna(), color='cyan')
    plt.title(f'Boxplot of {col}')
plt.tight_layout()
plt.show()

### Bivariate Analysis
* Distance vs Air Time, Departure Delay vs Arrival Delay, Distance vs Arrival Delay
* Use: Scatter plots, Correlation matrix

In [ ]:
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
sns.scatterplot(data=df, x='distance', y='air_time', alpha=0.3)
plt.title('Distance vs Air Time')

plt.subplot(1, 3, 2)
sns.scatterplot(data=df, x='dep_delay', y='arr_delay', alpha=0.3)
plt.title('Departure Delay vs Arrival Delay')

plt.subplot(1, 3, 3)
sns.scatterplot(data=df, x='distance', y='arr_delay', alpha=0.3)
plt.title('Distance vs Arrival Delay')
plt.tight_layout()
plt.show()

### Multivariate Analysis
* Delays, Distance, Air Time across Carriers
* Use: Pair plots, Heatmap

In [ ]:
# Sample data for pair plot (full dataset is too large)
df_sample = df[['dep_delay', 'arr_delay', 'distance', 'air_time', 'carrier']].dropna().sample(n=5000, random_state=42)
sns.pairplot(df_sample, hue='carrier', corner=True)
plt.show()

plt.figure(figsize=(10, 6))
numeric_cols = ['dep_delay', 'arr_delay', 'distance', 'air_time']
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='RdYlGn', fmt='.2f')
plt.title('Correlation Matrix Heatmap')
plt.show()

## Task 3: Handling Missing Data and Outliers
* Identify missing values
* Handle missing values using: Mean / Median / Mode
* Detect outliers using: Boxplots
* Explain the impact of outliers

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

# Handling missing values
df['dep_delay'].fillna(df['dep_delay'].median(), inplace=True)
df['arr_delay'].fillna(df['arr_delay'].median(), inplace=True)
df['air_time'].fillna(df['air_time'].median(), inplace=True)

print("\nMissing values after handling:")
print(df[['dep_delay', 'arr_delay', 'air_time']].isnull().sum())

print("\nImpact of Outliers:")
print("Outliers are extreme values that can significantly deviate results of the Mean and Standard Deviation. ")
print("In flight data, extreme delays (outliers) can skew average delay calculations and affect predictive models. ")
print("They may represent special events like weather disruptions or mechanical issues.")

## Task 4: Spread of Data
* Check data distribution: Normal / Skewed
* Calculate: Mean, Median, Standard deviation, Skewness, Kurtosis
* Interpret results

In [ ]:
spread_stats = pd.DataFrame({
    'Mean': df[['dep_delay', 'arr_delay', 'distance']].mean(),
    'Median': df[['dep_delay', 'arr_delay', 'distance']].median(),
    'Std Dev': df[['dep_delay', 'arr_delay', 'distance']].std(),
    'Skewness': df[['dep_delay', 'arr_delay', 'distance']].skew(),
    'Kurtosis': df[['dep_delay', 'arr_delay', 'distance']].kurt()
})
display(spread_stats)

print("\nInterpretation:")
print("Positive skewness in delays indicates right-skewed distribution with occasional very long delays.")
print("Distance shows more uniform distribution. High kurtosis in delays suggests presence of extreme outliers.")

## Task 5: Automating EDA using Python
* Use Pandas and NumPy functions: describe(), info(), isnull(), corr()
* Write reusable Python functions for EDA

In [ ]:
def perform_automated_eda(data):
    print("--- Data Information ---")
    data.info()
    print("\n--- Descriptive Statistics ---")
    display(data.describe())
    print("\n--- Correlation (Numeric Only) ---")
    display(data.corr(numeric_only=True))
    print("\n--- Null Count ---")
    print(data.isnull().sum())

perform_automated_eda(df)

## Task 6: Regression Analysis
* Identify: Dependent variable (Arrival Delay), Independent variables (Departure Delay, Distance)
* Perform: Simple Linear Regression
* Analyze: Covariance, Correlation

In [ ]:
print("Dependent variable: arr_delay (Arrival Delay)")
print("Independent variables: dep_delay (Departure Delay), distance")
print("\nCovariance Matrix:")
display(df[['arr_delay', 'dep_delay', 'distance']].cov(numeric_only=True))
print("\nCorrelation Matrix:")
display(df[['arr_delay', 'dep_delay', 'distance']].corr(numeric_only=True))

## Task 7 & Task 9: Supervised Learning – Regression Model
* Split dataset into: Training data, Testing data
* Build: (i) Simple Linear Regression, (ii) Multi Linear Regression, (iii) Logistic Regression
* Explain: Overfitting, Underfitting

In [ ]:
# Prepare data for regression (remove missing values)
df_clean = df[['dep_delay', 'distance', 'arr_delay', 'air_time']].dropna()

X = df_clean[['dep_delay', 'distance']]
y = df_clean['arr_delay']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# (i) Simple Linear Regression
slr = LinearRegression()
slr.fit(X_train[['dep_delay']], y_train)
print(f"Simple Linear Regression Test (R2): {slr.score(X_test[['dep_delay']], y_test):.4f}")

# (ii) Multi Linear Regression
mlr = LinearRegression()
mlr.fit(X_train, y_train)
print(f"Multi Linear Regression Test (R2): {mlr.score(X_test, y_test):.4f}")

# (iii) Logistic Regression (Classification task preparation)
# Create binary target: Is flight delayed? (arr_delay > 0)
df_clean['Is_Delayed'] = (df_clean['arr_delay'] > 0).astype(int)
X_c = df_clean[['dep_delay', 'distance', 'air_time']]
y_c = df_clean['Is_Delayed']
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_c, y_c, test_size=0.2, random_state=42)
lor = LogisticRegression(max_iter=1000)
lor.fit(X_train_c, y_train_c)

print("\nUnderfitting vs Overfitting Analysis:")
print(f"Train R2: {mlr.score(X_train, y_train):.4f}")
print(f"Test R2: {mlr.score(X_test, y_test):.4f}")
print("\nExplanation:")
print("- Underfitting: Model is too simple and fails to capture patterns (low train & test scores)")
print("- Overfitting: Model memorizes training data but fails on new data (high train, low test score)")
print("- Good Fit: Similar train and test scores indicate good generalization")

## Task 10, 11, 12: Classification Task and Model Evaluation
* Convert Arrival Delay -> Categorical Target (Delayed/On-time)
* Evaluate using Accuracy, Confusion Matrix, MSE, MAE, R² Score

In [ ]:
# Multi Regression Evaluation (Task 11/12)
y_pred_mlr = mlr.predict(X_test)
print("--- Regression Model Evaluation (Multi Linear) ---")
print(f"Mean Squared Error (MSE): {mean_squared_error(y_test, y_pred_mlr):.2f}")
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test, y_pred_mlr):.2f}")
print(f"R2 Score: {r2_score(y_test, y_pred_mlr):.4f}")

# Classification Evaluation (Task 10/11)
y_pred_lor = lor.predict(X_test_c)
print("\n--- Classification Performance (Logistic Regression) ---")
print(f"Accuracy Score: {accuracy_score(y_test_c, y_pred_lor):.4f}")
print("Confusion Matrix:")
cm = confusion_matrix(y_test_c, y_pred_lor)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: Flight Delay Classification')
plt.show()

## Task 13: Data Visualization Consolidation
* Plots: Univariate, Bivariate, Multivariate analysis
* Highlighting distribution normalization

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(df['arr_delay'].dropna(), fill=True, color='green')
plt.title('Normality Check: Arrival Delay Density Plot')
plt.xlabel('Arrival Delay (minutes)')
plt.show()

# Additional visualization: Delay by Carrier
plt.figure(figsize=(12, 6))
carrier_delays = df.groupby('carrier')['arr_delay'].mean().sort_values()
carrier_delays.plot(kind='barh', color='steelblue')
plt.title('Average Arrival Delay by Carrier')
plt.xlabel('Average Delay (minutes)')
plt.ylabel('Carrier')
plt.tight_layout()
plt.show()